In [2]:
import torch
from tqdm import tqdm
from torch import nn
from torch.nn import functional as F 

In [8]:

batch_size  = 4
block_size  = 8
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 300
learning_rate   = 1e-3 
max_iters       = 3000
eval_iter       = 200
n_emd           = 16 

In [18]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.tril   = torch.tril(torch.ones(block_size,block_size))
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [19]:
B,T,C = batch_size,block_size,n_emd
x = torch.randn(B,T,C)

In [20]:
model = Head(head_size=16)
model.to("cuda:0")

Head(
  (key): Linear(in_features=16, out_features=16, bias=False)
  (query): Linear(in_features=16, out_features=16, bias=False)
  (value): Linear(in_features=16, out_features=16, bias=False)
)

In [22]:
model.tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

- ❌ tril won’t move to the right device

- ❌ Won’t be saved in .state_dict()

In [24]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.zeros(block_size,block_size))) 
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [26]:
model = Head(head_size=16)
model.to("cuda:0")

Head(
  (key): Linear(in_features=16, out_features=16, bias=False)
  (query): Linear(in_features=16, out_features=16, bias=False)
  (value): Linear(in_features=16, out_features=16, bias=False)
)

In [27]:
model.tril

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0')

- ✅ It’s saved in state_dict()

- ✅ It moves to .cuda() / .to(device)